# CASAS Smart-Home Activity Dataset: Exploratory Data Analysis
**DA331 Big Data Analytics, Assignment 5 (Task 1)**

Dataset: CASAS Smart Home, free living, motion/door sensors with activity labels
([Zenodo 15708568](https://zenodo.org/records/15708568)). Each record is one ambient-sensor event
(PIR motion `ON/OFF`, magnetic door `OPEN/CLOSE`) with a timestamp, a sensor identifier named after its
location in the home (e.g. `BedroomAArea`, `KitchenA`), and the annotated activity of the resident.

**Tools:** `pandas` (vectorised regex parsing, categorical dtypes), `pyarrow`/Parquet (columnar cache),
`matplotlib`/`seaborn` (visualisation).

**Plan**
1. Load and normalise all files, then check the schema and memory use
2. Data quality: missing values, duplicates, ordering, time coverage
3. Sensors: types, most active sensors, message distribution
4. Temporal patterns: hour of day, day of week, daily volume, inter-event times
5. Activities: label distribution, episodes and durations, time-of-day profile, sensor-activity association
6. Activity transitions (these motivate the next-activity model)
7. Summary of findings

In [ ]:
import os, re, gzip, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", context="notebook")

# ---- Configuration ---------------------------------------------------------
# On Kaggle: "Add Input" -> the CASAS dataset; it is mounted under /kaggle/input.
DATA_DIR = Path(os.environ.get("CASAS_DATA_DIR",
                "/kaggle/input" if Path("/kaggle/input").exists() else "data"))
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
MAX_FILES = None                 # e.g. 5 to prototype on a few homes first
OTHER_LABELS = {"Other_Activity", "Other", "other", "None", "none", "NULL", ""}
print("DATA_DIR =", DATA_DIR, "| OUT_DIR =", OUT_DIR)

## Data loading

CASAS smart-home files are plain-text event logs, one sensor event per line:

```
<date> <time>  <sensor id> [<location/alias>]  <message>  [<activity label>]
2012-07-20 12:51:19.270391   BedroomAArea   ON   Sleep
```

Different releases (and the Kaggle mirror) differ slightly: tab vs space vs comma separated,
with/without a header, an optional extra location column, gz-compressed, and older releases
mark activities only as `Activity begin` / `Activity end`. The loader below auto-detects all
of these, uses **vectorised regex parsing** (no Python loop over rows), stores strings as
**categoricals** to keep memory small, and caches the result as **Parquet** so that re-runs are instant.

In [ ]:
STATE_RE = r"ON|OFF|OPEN|CLOSE|OPENED|CLOSED|PRESENT|ABSENT|TRUE|FALSE"
EVENT_RE = re.compile(  # date, time, rest-of-line
    r"^(?P<date>\d{4}-\d{2}-\d{2})[ T]+(?P<time>\d{1,2}:\d{2}(?::\d{2}(?:\.\d+)?)?)\S*\s+(?P<rest>.+)$")
REST_STATE_RE = (  # sensor [alias...] STATE [label...]
    rf"^(?P<sensor>\S+)(?:\s+(?P<location>\S+(?:\s+\S+)*?))?\s+(?P<message>{STATE_RE})(?:\s+(?P<activity>.*))?$")
REST_GENERIC_RE = r"^(?P<sensor>\S+)\s+(?P<message>\S+)(?:\s+(?P<activity>.*))?$"   # e.g. T001 21.5

COLUMN_ALIASES = {
    "timestamp": ["timestamp", "datetime", "date_time", "time_stamp", "ts"],
    "date": ["date", "day"], "time": ["time"],
    "sensor": ["sensor", "sensor_id", "sensorid", "sensor_name", "device", "item"],
    "location": ["location", "sensor_location", "alias", "room", "sensor_alias"],
    "message": ["message", "state", "value", "status", "sensor_state", "sensor_status", "event", "reading"],
    "activity": ["activity", "label", "activity_label", "annotation", "class", "activity_name"],
}
DATA_EXT = {".txt", ".csv", ".tsv", ".dat", ".gz", ".parquet", ""}
SKIP_NAME = re.compile(r"readme|license|licence|description|metadata|\.md$|\.json$|\.pdf$|\.py$|\.ipynb$", re.I)
GENERIC_STEMS = {"data", "ann", "annotated", "raw", "rawdata", "events", "labels", "sensor", "sensors"}


def find_data_files(root):
    files = [p for p in Path(root).rglob("*")
             if p.is_file() and p.suffix.lower() in DATA_EXT and not SKIP_NAME.search(p.name)
             and p.stat().st_size > 0 and "/kaggle/working" not in str(p)]
    return sorted(files)


def home_name(p):
    stem = p.name.split(".")[0]
    return p.parent.name if stem.lower() in GENERIC_STEMS else stem


def _open_text(p):
    return gzip.open(p, "rt", errors="replace") if p.suffix.lower() == ".gz" else open(p, "r", errors="replace")


def _map_columns(cols):
    low = {c: re.sub(r"[^a-z_]", "", c.strip().lower().replace(" ", "_")) for c in cols}
    out = {}
    for std, names in COLUMN_ALIASES.items():
        for c, l in low.items():
            if l in names and c not in out.values():
                out[std] = c
                break
    return out


def _from_table(df):
    """Normalise a table that has a header (CSV / Parquet) to the standard schema."""
    m = _map_columns(df.columns)
    if "timestamp" in m:
        ts = df[m["timestamp"]].astype(str)
    elif "date" in m and "time" in m:
        ts = df[m["date"]].astype(str) + " " + df[m["time"]].astype(str)
    else:
        raise ValueError(f"Cannot find timestamp columns in {list(df.columns)}")
    if "sensor" not in m or "message" not in m:
        raise ValueError(f"Cannot find sensor/message columns in {list(df.columns)}")
    out = pd.DataFrame({"timestamp": ts, "sensor": df[m["sensor"]].astype(str),
                        "message": df[m["message"]].astype(str)})
    out["location"] = df[m["location"]].astype(str) if "location" in m else np.nan
    out["activity"] = df[m["activity"]].astype(str) if "activity" in m else np.nan
    return out


def _from_lines(lines):
    """Vectorised parse of header-less CASAS text lines."""
    s = pd.Series(lines, dtype="string").str.replace(r"[,\t;]+", " ", regex=True).str.strip()
    ev = s.str.extract(EVENT_RE)
    ev = ev[ev["date"].notna()]
    rest = ev["rest"]
    parsed = rest.str.extract(REST_STATE_RE, flags=re.I)
    miss = parsed["sensor"].isna()
    if miss.any():
        gen = rest[miss].str.extract(REST_GENERIC_RE)
        parsed.loc[miss, ["sensor", "message", "activity"]] = gen[["sensor", "message", "activity"]].values
    return pd.DataFrame({"timestamp": ev["date"] + " " + ev["time"].fillna("00:00:00"),
                         "sensor": parsed["sensor"], "location": parsed["location"],
                         "message": parsed["message"].str.upper(), "activity": parsed["activity"]})


def _expand_begin_end(df):
    """Old CASAS format: labels only as 'X begin' / 'X end' -> label every event in between."""
    act = df["activity"].fillna("").astype(str).str.strip()
    mark = act.str.extract(r"^(?P<name>.+?)\s+(?P<kind>begin|end)$", flags=re.I)
    if mark["kind"].notna().sum() == 0 or (act[mark["kind"].isna()] != "").mean() > 0.01:
        return df                                   # already one label per event
    lab = pd.Series(np.nan, index=df.index, dtype=object)     # label of the marker row itself
    after = pd.Series(np.nan, index=df.index, dtype=object)   # what is active after the marker
    active = []
    for idx, name, kind in mark.dropna().itertuples():
        if kind.lower() == "begin":
            active.append(name)
        elif name in active:
            active.remove(name)
        lab[idx] = name
        after[idx] = active[-1] if active else "Other_Activity"
    cur = lab.fillna(after.shift(1).ffill())
    df = df.copy()
    df["activity"] = cur.ffill().fillna("Other_Activity")
    return df


def load_file(p):
    if p.suffix.lower() == ".parquet":
        df = _from_table(pd.read_parquet(p))
    else:
        with _open_text(p) as f:
            head = [f.readline() for _ in range(5)]
        first = next((l for l in head if l.strip()), "")
        has_header = bool(first) and not re.match(r"^\s*\d{4}-\d{2}-\d{2}", first) and bool(
            re.search(r"sensor|time|date|activity|label|message|state", first, re.I))
        if has_header:
            sep = max([",", "\t", ";"], key=first.count)
            df = _from_table(pd.read_csv(p, sep=sep, dtype=str, compression="infer"))
        else:
            with _open_text(p) as f:
                df = _from_lines(f.read().splitlines())
    df["timestamp"] = pd.to_datetime(df["timestamp"], format="mixed", errors="coerce")
    df = df.dropna(subset=["timestamp", "sensor"])
    df["activity"] = df["activity"].where(df["activity"].notna(), np.nan)
    if df["activity"].notna().any():
        df["activity"] = df["activity"].astype(str).str.strip().replace({"nan": np.nan, "": np.nan})
        df = _expand_begin_end(df)
    df.insert(0, "home", home_name(p))
    return df


def sensor_type(msg):
    m = msg.astype(str).str.upper()
    num = pd.to_numeric(m, errors="coerce").notna()
    return np.select([m.isin(["ON", "OFF"]), m.isin(["OPEN", "CLOSE", "OPENED", "CLOSED"]), num],
                     ["motion", "door", "numeric"], "other")


def load_casas(data_dir, max_files=None, cache=None):
    if cache is not None and Path(cache).exists():
        print("Loading cached", cache)
        return pd.read_parquet(cache)
    files = find_data_files(data_dir)[:max_files]
    if not files:
        raise FileNotFoundError(f"No data files found under {data_dir}. Attach the dataset (Kaggle: Add Input).")
    parts, t0 = [], time.time()
    for p in files:
        try:
            d = load_file(p)
        except Exception as e:                      # keep going if one odd file is present
            print(f"  skip {p.name}: {e}"); continue
        if len(d) == 0:
            print(f"  skip {p.name}: no parsable events"); continue
        parts.append(d)
        print(f"  {p.relative_to(data_dir)}: {len(d):,} events")
    df = pd.concat(parts, ignore_index=True)
    df = df.sort_values(["home", "timestamp"], kind="stable").reset_index(drop=True)
    df["sensor_type"] = sensor_type(df["message"])
    for c in ["home", "sensor", "location", "message", "activity", "sensor_type"]:
        df[c] = df[c].astype("category")
    print(f"Loaded {len(df):,} events from {df['home'].nunique()} home(s) in {time.time()-t0:.1f}s")
    if cache is not None:
        try:
            df.to_parquet(cache, index=False)
        except Exception as e:
            print(f"Parquet cache not written ({type(e).__name__}); continuing without cache")
    return df

In [ ]:
files = find_data_files(DATA_DIR)
inv = pd.DataFrame({"file": [str(p.relative_to(DATA_DIR)) for p in files],
                    "home": [home_name(p) for p in files],
                    "size_MB": [p.stat().st_size / 1e6 for p in files]})
print(f"{len(inv)} data files, {inv.size_MB.sum():,.1f} MB in total")
inv.sort_values("size_MB", ascending=False).head(20)

In [ ]:
df = load_casas(DATA_DIR, max_files=MAX_FILES, cache=OUT_DIR / f"casas_events_{MAX_FILES or 'all'}.parquet")
df.head(10)

## 1. Schema, size and memory

In [ ]:
print("Shape:", df.shape)
print(df.dtypes, "\n")
mem = df.memory_usage(deep=True).sum() / 1e6
mem_obj = df.astype({c: "object" for c in df.select_dtypes("category").columns}).memory_usage(deep=True).sum() / 1e6
print(f"Memory with categoricals: {mem:,.1f} MB (plain object strings would be {mem_obj:,.1f} MB)")
df.describe(include="all").T

## 2. Data quality

In [ ]:
quality = pd.DataFrame({
    "missing": df.isna().sum(),
    "missing_%": (df.isna().mean() * 100).round(2),
    "n_unique": df.nunique(),
})
display(quality)
dups = df.duplicated(["home", "timestamp", "sensor", "message"]).sum()
print(f"Exact duplicate events: {dups:,} ({dups/len(df):.2%})")
unordered = (df.groupby("home", observed=True)["timestamp"].diff().dt.total_seconds() < 0).sum()
print("Out-of-order timestamps after sort:", unordered)

In [ ]:
cov = df.groupby("home", observed=True).agg(
    events=("timestamp", "size"), start=("timestamp", "min"), end=("timestamp", "max"),
    sensors=("sensor", "nunique"), activities=("activity", "nunique"),
    labelled_share=("activity", lambda s: s.notna().mean()))
cov["days"] = (cov["end"] - cov["start"]).dt.total_seconds() / 86400
cov["events_per_day"] = cov["events"] / cov["days"].clip(lower=1)
cov.sort_values("events", ascending=False).head(20)

In [ ]:
# Data gaps: days with no events inside each home's observation window (sensor/network outages)
day_counts = df.groupby(["home", df["timestamp"].dt.floor("D")], observed=True).size()
gaps = {}
for h, s in day_counts.groupby(level=0):
    s = s.droplevel(0)
    full = pd.date_range(s.index.min(), s.index.max(), freq="D")
    gaps[h] = len(full.difference(s.index))
print("Homes with missing days:", {k: v for k, v in gaps.items() if v} or "none")

## 3. Sensors

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
df["sensor_type"].value_counts().plot.bar(ax=ax[0], color="C0"); ax[0].set_title("Events by sensor type")
df["message"].value_counts().head(10).plot.bar(ax=ax[1], color="C1"); ax[1].set_title("Top messages")
df["sensor"].value_counts().head(20).plot.barh(ax=ax[2], color="C2"); ax[2].invert_yaxis()
ax[2].set_title("20 most active sensors")
plt.tight_layout(); plt.show()

In [ ]:
# Sensor names encode the room (BedroomAArea, KitchenA, ...). Strip the trailing letter/Area suffix to get a room.
# If the file has a separate location/alias column (e.g. "M007 Kitchen ON"), use that instead.
base = df["location"] if df["location"].notna().mean() > 0.5 else df["sensor"]
room = base.astype(str).str.replace(r"(Area)?[A-Z]?\d*$", "", regex=True)
room = room.where(room.str.len() > 1, base.astype(str))           # fall back for M001-style ids
df["room"] = room.astype("category")
df["room"].value_counts().head(15).plot.barh(figsize=(7, 5), title="Events by room (derived from sensor name)")
plt.gca().invert_yaxis(); plt.show()

## 4. Temporal patterns

In [ ]:
ts = df["timestamp"]
fig, ax = plt.subplots(1, 2, figsize=(15, 4))
ts.dt.hour.value_counts().sort_index().plot.bar(ax=ax[0], color="C0")
ax[0].set(title="Sensor events by hour of day", xlabel="hour")
ts.dt.day_name().value_counts().reindex(
    ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]).plot.bar(ax=ax[1], color="C1")
ax[1].set(title="Sensor events by day of week"); plt.tight_layout(); plt.show()

hw = pd.crosstab(ts.dt.dayofweek, ts.dt.hour)
hw.index = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"][: len(hw)]
plt.figure(figsize=(15, 3.5)); sns.heatmap(hw, cmap="viridis"); plt.title("Events: weekday x hour"); plt.show()

In [ ]:
top_homes = df["home"].value_counts().head(5).index
daily = (df[df["home"].isin(top_homes)]
         .groupby(["home", df["timestamp"].dt.floor("D")], observed=True).size().unstack(0))
daily.plot(figsize=(15, 4), alpha=.8, title="Daily event volume (top-5 homes by size)"); plt.ylabel("events/day"); plt.show()

In [ ]:
dt_s = df.groupby("home", observed=True)["timestamp"].diff().dt.total_seconds().dropna()
print(dt_s.describe(percentiles=[.5, .9, .99]).round(2))
plt.figure(figsize=(8, 4))
plt.hist(np.log10(dt_s.clip(lower=1e-3)), bins=100, color="C3")
plt.xlabel("log10(seconds between consecutive events)"); plt.title("Inter-event time distribution"); plt.show()

## 5. Activities

In [ ]:
lab = df["activity"].astype(str).where(df["activity"].notna(), "<unlabelled>")
act_counts = lab.value_counts()
print("Distinct activity labels:", df["activity"].nunique())
fig, ax = plt.subplots(figsize=(9, max(4, 0.3 * len(act_counts))))
(act_counts / act_counts.sum() * 100).sort_values().plot.barh(ax=ax, color="C4")
ax.set(xlabel="% of sensor events", title="Activity label distribution (event level)"); plt.show()

Because one activity instance produces many sensor events, event counts over-weight long activities
(e.g. *Sleep*, *Relax*). We therefore also look at **episodes**: maximal runs of consecutive events
with the same label. The same episode representation is the input to the next-activity model.

In [ ]:
def build_episodes(df, drop_other=True, merge_gap_min=10):
    """Collapse the event stream into activity episodes (runs of the same label).

    If drop_other, 'Other_Activity' runs are removed and neighbouring runs of the same
    activity that are closer than merge_gap_min minutes are merged back together
    (e.g. Sleep - a few unlabelled events - Sleep  ->  one Sleep episode).
    """
    d = df.loc[df["activity"].notna(), ["home", "timestamp", "activity", "sensor"]].copy()
    for c in ["home", "activity", "sensor"]:
        d[c] = d[c].astype(str)
    new = (d["activity"] != d["activity"].shift()) | (d["home"] != d["home"].shift())
    ep = d.groupby(new.cumsum(), sort=False).agg(
        home=("home", "first"), activity=("activity", "first"),
        start=("timestamp", "first"), end=("timestamp", "last"),
        n_events=("timestamp", "size"), last_sensor=("sensor", "last"),
        n_sensors=("sensor", "nunique")).reset_index(drop=True)
    if drop_other:
        ep = ep[~ep["activity"].isin(OTHER_LABELS)].reset_index(drop=True)
        gap = (ep["start"] - ep["end"].shift()).dt.total_seconds() / 60
        new = ((ep["activity"] != ep["activity"].shift()) | (ep["home"] != ep["home"].shift())
               | (gap > merge_gap_min))
        ep = ep.groupby(new.cumsum(), sort=False).agg(
            home=("home", "first"), activity=("activity", "first"), start=("start", "min"),
            end=("end", "max"), n_events=("n_events", "sum"), last_sensor=("last_sensor", "last"),
            n_sensors=("n_sensors", "max")).reset_index(drop=True)
    ep["duration_min"] = (ep["end"] - ep["start"]).dt.total_seconds() / 60
    return ep

In [ ]:
ep_all = build_episodes(df, drop_other=False)
ep = build_episodes(df, drop_other=True)
print(f"Episodes incl. Other: {len(ep_all):,} | labelled episodes (Other removed/merged): {len(ep):,}")
share_other = ep_all["activity"].isin(OTHER_LABELS).mean()
print(f"Share of 'Other' episodes: {share_other:.1%}")
ep_stats = ep.groupby("activity").agg(episodes=("activity", "size"),
                                      median_min=("duration_min", "median"),
                                      mean_min=("duration_min", "mean"),
                                      median_events=("n_events", "median")).sort_values("episodes", ascending=False)
ep_stats.round(1)

In [ ]:
order = ep_stats.index[:20]
plt.figure(figsize=(12, 6))
sns.boxplot(data=ep[ep["activity"].isin(order)], y="activity", x="duration_min", order=order,
            showfliers=False, color="C0")
plt.xscale("symlog"); plt.title("Episode duration by activity (minutes, symlog)"); plt.show()

In [ ]:
prof = pd.crosstab(ep["activity"], ep["start"].dt.hour, normalize="index").loc[order]
plt.figure(figsize=(15, 0.35 * len(order) + 2))
sns.heatmap(prof, cmap="magma_r", cbar_kws={"label": "share of episodes"})
plt.title("When does each activity start? (hour of day, row-normalised)"); plt.xlabel("hour"); plt.show()

In [ ]:
lab_ev = df[df["activity"].notna() & ~df["activity"].isin(OTHER_LABELS)]
top_rooms = lab_ev["room"].value_counts().head(15).index
sa = pd.crosstab(lab_ev["activity"].astype(str), lab_ev["room"].astype(str), normalize="index")
sa = sa.reindex(index=[a for a in order if a in sa.index], columns=[r for r in top_rooms if r in sa.columns])
plt.figure(figsize=(14, 0.35 * len(sa) + 2))
sns.heatmap(sa, cmap="Blues", cbar_kws={"label": "share of the activity's events"})
plt.title("Where do activities happen? (activity x room)"); plt.show()

## 6. Activity transitions

In [ ]:
nxt = ep.groupby("home")["activity"].shift(-1)
trans = pd.crosstab(ep["activity"], nxt, normalize="index")
trans = trans.reindex(index=order, columns=order).fillna(0)
plt.figure(figsize=(13, 10))
sns.heatmap(trans, cmap="rocket_r", annot=len(order) <= 15, fmt=".2f")
plt.title("P(next activity | current activity), Other removed"); plt.xlabel("next"); plt.ylabel("current"); plt.show()

top1 = trans.max(axis=1)
print(f"Mean probability of the single most likely successor: {top1.mean():.2f}")
print("Most predictable transitions:")
print(pd.DataFrame({"next": trans.idxmax(axis=1), "p": top1}).sort_values("p", ascending=False).head(10).round(2))

In [ ]:
# How much does knowing the time of day add on top of the current activity? (conditional entropy, bits)
def cond_entropy(x, y):
    joint = pd.crosstab(x, y); p = joint / joint.values.sum()
    px = p.sum(axis=1)
    return float(-(p * np.log2((p / px.values[:, None]).replace(0, np.nan))).sum().sum())
e = ep.assign(next=nxt).dropna(subset=["next"])
H_next = cond_entropy(pd.Series(0, index=e.index), e["next"])
H_cur = cond_entropy(e["activity"], e["next"])
H_cur_hour = cond_entropy(e["activity"] + "@" + (e["end"].dt.hour // 3).astype(str), e["next"])
print(f"H(next) = {H_next:.2f} bits | H(next|current) = {H_cur:.2f} | H(next|current, 3h time-bin) = {H_cur_hour:.2f}")

## 7. Summary

In [ ]:
summary = {
    "homes": df["home"].nunique(),
    "events": len(df),
    "period": f'{df["timestamp"].min():%Y-%m-%d} to {df["timestamp"].max():%Y-%m-%d}',
    "sensors": df["sensor"].nunique(),
    "activity labels": df["activity"].nunique(),
    "share labelled (not Other)": f'{(df["activity"].notna() & ~df["activity"].isin(OTHER_LABELS)).mean():.1%}',
    "labelled episodes": len(ep),
    "most frequent episode": ep["activity"].value_counts().idxmax(),
    "median inter-event time (s)": round(float(dt_s.median()), 2),
    "H(next) / H(next|cur) / H(next|cur,time) bits": f"{H_next:.2f} / {H_cur:.2f} / {H_cur_hour:.2f}",
}
pd.Series(summary, name="value").to_frame()

**Key findings** (check these against the numbers printed above for your data):

* The data is a long, irregular **event stream**. Events come in bursts while someone moves and go
  quiet for hours at night, so inter-event times are heavy-tailed. Fixed time windows would mostly be
  empty, which is why we model **activity episodes** instead of raw events.
* Motion (`ON/OFF`) events dominate and door events are rare. Sensor names encode rooms, which makes
  the room a strong cue for the activity (see the activity x room heatmap).
* Activity volume follows a clear **circadian rhythm**: little activity at night apart from
  *Sleep* / *Bed_Toilet_Transition*, and peaks in the morning and evening around meals and hygiene.
* Labels are **imbalanced**. `Other_Activity` makes up a large share of events, and among the named
  activities a few (sleep, relax, hygiene, meals) dominate. The model is therefore scored with
  macro-F1 as well as accuracy.
* The transition matrix is far from uniform, and conditioning on the current activity (and the hour)
  clearly lowers the entropy of the next activity. A Markov / time-aware model is a sensible baseline
  for next-activity prediction (Task 2).